### This code is a new branch where CSAT data that use is after analysis (Taking Out The Unnecessary Data)

In [1]:
import gspread
import pandas as pd
from datetime import date
from google.oauth2.service_account import Credentials

## Pipeline A

In [2]:
SERVICE_ACCOUNT_FILE = '../../key/credentials.json'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets',
          'https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
client = gspread.authorize(creds)

sheet = client.open("Chatbot Ops Report").worksheet("[CSAT After Take Out]")
data = sheet.get_all_values()

df = pd.DataFrame(data[1:], columns=data[0])
df.to_csv('../../raw_data/csat_2.csv', index=False)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 968 entries, 0 to 967
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Date               968 non-null    object
 1   Total Good Survey  968 non-null    object
 2   Total Bad Survey   968 non-null    object
 3   Total Responden    968 non-null    object
 4   Total Rating       968 non-null    object
 5   CSAT [Before]      968 non-null    object
 6   CSAT [After]       968 non-null    object
 7   %                  968 non-null    object
 8                      968 non-null    object
 9   Date               968 non-null    object
 10  Total Good Survey  968 non-null    object
 11  Total Bad Survey   968 non-null    object
 12  Total Responden    968 non-null    object
 13  Total Rating       968 non-null    object
 14  CSAT               968 non-null    object
dtypes: object(15)
memory usage: 113.6+ KB


In [4]:
# Copying the DF
df_clean = (
    df.copy()
    .iloc[:, [0,3,4,5,6]]
    .drop_duplicates(subset='Date')
)

# Changing the Date data type
df_clean['Date'] = pd.to_datetime(
    df_clean['Date'], format='%d-%b-%Y', errors='coerce'
)

# dropping NA
df_clean = df_clean.dropna(subset=['Date'])

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 966 entries, 0 to 967
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             966 non-null    datetime64[ns]
 1   Total Responden  966 non-null    object        
 2   Total Rating     966 non-null    object        
 3   CSAT [Before]    966 non-null    object        
 4   CSAT [After]     966 non-null    object        
dtypes: datetime64[ns](1), object(4)
memory usage: 45.3+ KB


In [5]:
df_clean

,Date,Total Responden,Total Rating,CSAT [Before],CSAT [After]
0,2024-03-01,59,196,3.32,3.68
1,2024-03-02,49,162,3.31,3.67
2,2024-03-03,40,152,3.80,3.95
3,2024-03-04,42,152,3.62,3.97
4,2024-03-05,54,190,3.52,3.65
...,...,...,...,...,...
962,2026-10-28,44,158,3.59,4.00
963,2026-10-29,40,157,3.93,4.16
964,2026-10-30,35,133,3.80,4.16
965,2026-10-31,46,172,3.74,4.10


## Pipeline B

In [6]:
df_b = pd.read_csv('../../dataset_kula/csat/akulaku_superset_presto_hanmm_ozy.prazuganda_1755663318_8_19.csv')
df_b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11949 entries, 0 to 11948
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   business                           11949 non-null  object 
 1   id                                 11949 non-null  object 
 2   currentAccount                     11949 non-null  object 
 3   userId                             11949 non-null  int64  
 4   shopid                             11949 non-null  int64  
 5   summary_type                       0 non-null      float64
 6   summary_classify                   0 non-null      float64
 7   ticketid                           4805 non-null   object 
 8   Category 1                         4492 non-null   object 
 9   Category 2                         4146 non-null   object 
 10  Category 3                         3912 non-null   object 
 11  summary_text                       0 non-null      flo

In [7]:
# Filtering the currentAccount into robot only
df_filtered = df_b[df_b['currentAccount'] == '3-1-robot']

# Cleaning and Counting
score = df_filtered['score'].dropna()

total_responden = score.count()
total_rating = int(score.sum())
csat = round(total_rating / total_responden, 2) if total_responden > 0 else 0

# Good survey and bad survey counting
good_survey = score[score >= 3].count()
bad_survey = score[score < 3].count()

In [8]:
"""
-- Not necessary at the moment --

# Checking if the yesterday data already exists in the DataFrame
yesterday = pd.Timestamp(date.today() - pd.Timedelta(days=1))

if (df_clean['Date'].iloc[-1].normalize != yesterday.normalize()):
    new_row = pd.DataFrame([{
        'Date': yesterday,
        'Total Good Survey': good_survey,
        'Total Bad Survey': bad_survey,
        'Total Responden': total_responden,
        'Total Rating': total_rating,
        'CSAT': csat
    }])

    df_clean = pd.concat([df_clean, new_row], ignore_index=True)

"""

"\n-- Not necessary at the moment --\n\n# Checking if the yesterday data already exists in the DataFrame\nyesterday = pd.Timestamp(date.today() - pd.Timedelta(days=1))\n\nif (df_clean['Date'].iloc[-1].normalize != yesterday.normalize()):\n    new_row = pd.DataFrame([{\n        'Date': yesterday,\n        'Total Good Survey': good_survey,\n        'Total Bad Survey': bad_survey,\n        'Total Responden': total_responden,\n        'Total Rating': total_rating,\n        'CSAT': csat\n    }])\n\n    df_clean = pd.concat([df_clean, new_row], ignore_index=True)\n\n"

In [9]:
df_clean.to_csv('../../dataset_kula/csat_takeout.csv')

## Analisis CSAT robot Oktober 2026

Sumber: `dataset_kula/csat_aug_oct_2026_source.csv`, ekspor harian yang mencakup Agustus, September, dan Oktober 2026. Kolom kiri berisi **before takeout**; kolom kanan berisi **after takeout**. CSAT = total rating / total responden pada kolom kanan (skala 1–5), bukan rata-rata CSAT harian. Semua bulan dihitung penuh. Minggu Oktober: 1–7, 8–14, 15–21, 22–28, 29–31. Klasifikasi `dataset_kula/bad_survey.csv` menunjukkan pola keluhan, bukan bukti sebab-akibat atau insiden yang terkonfirmasi.

In [10]:
from pathlib import Path
import calendar
import pandas as pd
from IPython.display import display, Markdown
from plotly.subplots import make_subplots
import plotly.graph_objects as go

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'dataset_kula' / 'csat_aug_oct_2026_source.csv').exists())
source = root / 'dataset_kula' / 'csat_aug_oct_2026_source.csv'
# Ekspor memiliki dua blok berheader sama. Baca posisi kolom agar before dan after tidak tertukar.
columns = ['Date', 'Good Before', 'Bad Before', 'Respondents Before', 'Rating Before',
           'CSAT Before', 'CSAT After Shown', 'Increase', 'Separator', 'Date After',
           'Good After', 'Bad After', 'Respondents After', 'Rating After', 'CSAT After']
csat_data = pd.read_csv(source, header=0, names=columns, dtype=str)
csat_data['Date'] = pd.to_datetime(csat_data['Date'], format='%d-%b-%y', errors='coerce')
csat_data['Date After'] = pd.to_datetime(csat_data['Date After'], format='%d-%b-%y', errors='coerce')
for column in ['Good After', 'Bad After', 'Respondents After', 'Rating After',
               'Respondents Before', 'Rating Before']:
    csat_data[column] = pd.to_numeric(csat_data[column], errors='coerce')
analysis_year = 2026
months = {8: 'Agustus', 9: 'September', 10: 'Oktober'}
csat_data = csat_data[(csat_data['Date'].dt.year == analysis_year) &
                      (csat_data['Date'].dt.month.isin(months))].copy()
if csat_data.empty or csat_data['Date'].duplicated().any():
    raise ValueError('Data kosong atau tanggal duplikat; periksa ekspor harian')
if not csat_data['Date'].equals(csat_data['Date After']):
    raise ValueError('Tanggal pada blok before dan after berbeda')
if csat_data[['Respondents After', 'Rating After', 'Respondents Before',
              'Rating Before', 'Good After', 'Bad After']].isna().any().any():
    raise ValueError('Jumlah responden, rating, atau survei kosong')
if not (csat_data['Good After'] + csat_data['Bad After'] == csat_data['Respondents After']).all():
    raise ValueError('Good + bad tidak sama dengan responden after')
for month in months:
    dates = csat_data.loc[csat_data['Date'].dt.month == month, 'Date']
    if set(dates.dt.day) != set(range(1, calendar.monthrange(analysis_year, month)[1] + 1)):
        raise ValueError(f'Tanggal {months[month]} tidak lengkap; jangan tampilkan sebagai bulan penuh')
csat_data['Bulan'] = csat_data['Date'].dt.month.map(months)
monthly = csat_data.groupby('Bulan', sort=False).agg(
    hari=('Date', 'nunique'), responden=('Respondents After', 'sum'),
    rating=('Rating After', 'sum'), good=('Good After', 'sum'), bad=('Bad After', 'sum'),
    responden_before=('Respondents Before', 'sum'), rating_before=('Rating Before', 'sum'))
monthly = monthly.reindex(list(months.values()))
monthly['CSAT'] = monthly['rating'] / monthly['responden']
monthly['CSAT Before'] = monthly['rating_before'] / monthly['responden_before']
monthly['Proporsi Bad'] = 100 * monthly['bad'] / monthly['responden']
october = csat_data[csat_data['Date'].dt.month == 10].copy()
october['Minggu'] = (october['Date'].dt.day.sub(1) // 7).add(1)
weekly = october.groupby('Minggu').agg(
    hari=('Date', 'nunique'), responden=('Respondents After', 'sum'),
    rating=('Rating After', 'sum'), good=('Good After', 'sum'), bad=('Bad After', 'sum'))
weekly['CSAT'] = weekly['rating'] / weekly['responden']
weekly['rentang'] = [f'{(week - 1) * 7 + 1}-{min(week * 7, 31)} Okt'
                     for week in weekly.index]
display(Markdown('**Cakupan:** Agustus, September, Oktober 2026 penuh. '
                 'CSAT utama memakai jumlah rating / responden after takeout.'))
display(monthly[['hari', 'responden_before', 'rating_before', 'CSAT Before',
                 'responden', 'rating', 'CSAT', 'good', 'bad', 'Proporsi Bad']].round(3))
display(weekly[['rentang', 'hari', 'responden', 'rating', 'good', 'bad', 'CSAT']].round(3))
assert (monthly.loc['Oktober', ['responden_before', 'rating_before', 'responden',
                                 'rating', 'good', 'bad']] ==
        [1288, 5070, 1197, 4972, 997, 200]).all()
assert (weekly.loc[4, ['responden', 'rating']] == [258, 1035]).all()

**Cakupan:** Agustus, September, Oktober 2026 penuh. CSAT utama memakai jumlah rating / responden after takeout.

,hari,responden_before,rating_before,CSAT Before,responden,rating,CSAT,good,bad,Proporsi Bad
Bulan,,,,,,,,,,
Agustus,31,1211.0,4797.0,3.961,1098.0,4673.0,4.256,930.0,168.0,15.301
September,30,1309.0,5042.0,3.852,1213.0,4934.0,4.068,971.0,242.0,19.951
Oktober,31,1288.0,5070.0,3.936,1197.0,4972.0,4.154,997.0,200.0,16.708


,rentang,hari,responden,rating,good,bad,CSAT
Minggu,,,,,,,
1,1-7 Okt,7,326.0,1327.0,266.0,60.0,4.071
2,8-14 Okt,7,259.0,1098.0,221.0,38.0,4.239
3,15-21 Okt,7,246.0,1065.0,214.0,32.0,4.329
4,22-28 Okt,7,258.0,1035.0,205.0,53.0,4.012
5,29-31 Okt,3,108.0,447.0,91.0,17.0,4.139


In [11]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    '3 Months CSAT Comparison', 'Weekly October CSAT'))
fig.add_trace(go.Bar(x=monthly.index.tolist(), y=monthly['CSAT'].round(3),
    text=monthly['CSAT'].map(lambda v: f'{v:.2f}'), textposition='outside',
    customdata=monthly['responden'],
    hovertemplate='%{x}: %{y:.3f}<br>Responden after: %{customdata:,.0f}<extra></extra>',
    marker_color=['#9caaad', '#9caaad', '#106b62'], showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=weekly['rentang'], y=weekly['CSAT'].round(3),
    mode='lines+markers+text', text=weekly['CSAT'].map(lambda v: f'{v:.2f}'),
    textposition='top center', line=dict(color='#106b62', width=3), marker=dict(size=10),
    customdata=weekly['responden'],
    hovertemplate='%{x}: %{y:.3f}<br>Responden after: %{customdata:,.0f}<extra></extra>',
    showlegend=False), row=1, col=2)
fig.update_yaxes(range=[0, 5.3], title_text='Average Rating', row=1, col=1)
fig.update_yaxes(range=[0, 5.3], title_text='Average Rating', row=1, col=2)
fig.update_layout(title='CSAT | August-October 2026',
    template='plotly_white', height=460, width=1050, margin=dict(t=95, b=85))
fig.add_annotation(x=0.99, y=-0.23, xref='paper', yref='paper', showarrow=False,
    text='Sumber: csat_aug_oct_2026_source.csv | data after takeout',
    font=dict(size=11, color='#475569'), xanchor='right')
fig.show(renderer='plotly_mimetype')

In [12]:
qc = pd.read_csv(root / 'dataset_kula' / 'bad_survey.csv', low_memory=False)
qc['Conversation Start Time'] = pd.to_datetime(qc['Conversation Start Time'], errors='coerce')
qc_oct = qc[(qc['Conversation Start Time'].dt.year == analysis_year) &
            (qc['Conversation Start Time'].dt.month == 10)].copy()
qc_oct['Minggu'] = (qc_oct['Conversation Start Time'].dt.day.sub(1) // 7).add(1)
qc_coverage = qc_oct.groupby('Minggu').agg(
    catatan=('QC Result', 'size'), hari_terakhir=('Conversation Start Time', 'max'))
qc_counts = pd.crosstab(qc_oct['Minggu'], qc_oct['QC Result']).reindex(weekly.index, fill_value=0)
display(Markdown('### Bukti QC bad survey menurut minggu (cakupan dapat tertinggal)'))
display(qc_coverage)
display(qc_counts)
latest_qc = qc_oct['Conversation Start Time'].max()
if pd.isna(latest_qc):
    display(Markdown('**Batas QC:** tidak ada data QC Oktober; penyebab belum dapat dinilai.'))
elif latest_qc.day < 31:
    display(Markdown(f'**Batas QC:** percakapan terakhir {latest_qc:%d %B %Y}; '
                     f'tidak ada klasifikasi untuk {latest_qc.day + 1}-31 Oktober. '
                     'Jangan atribusikan penurunan di tanggal tersebut pada kategori QC yang belum diperiksa.'))
drops = weekly['CSAT'].diff().dropna()
for week, change in drops[drops < 0].items():
    categories = qc_counts.loc[week].sort_values(ascending=False).head(4)
    detail = ', '.join(f'{name}: {count}' for name, count in categories.items() if count > 0)
    count = int(qc_coverage.loc[week, 'catatan']) if week in qc_coverage.index else 0
    display(Markdown(f'**Minggu {week} ({weekly.loc[week, "rentang"]}):** '
                     f'CSAT turun {abs(change):.2f} poin dibanding minggu sebelumnya; '
                     f'{count} catatan QC tersedia. Klasifikasi: {detail or "belum tersedia"}. '
                     'Ini korelasi deskriptif, bukan penyebab yang terkonfirmasi.'))
daily_oct = october.assign(CSAT=october['Rating After'] / october['Respondents After'])
lowest = daily_oct.nsmallest(3, 'CSAT')
display(Markdown('**Tiga tanggal CSAT terendah** (periksa volume dan QC sebelum menyimpulkan):'))
display(lowest[['Date', 'Respondents After', 'CSAT']].round(3))

### Bukti QC bad survey menurut minggu (cakupan dapat tertinggal)

,catatan,hari_terakhir
Minggu,,
1,196,2026-10-07
2,127,2026-10-14
3,122,2026-10-21
4,20,2026-10-22


QC Result,Answer Robot Mistake,Chat Customer No Clear,Information Not Meet Customer Expectations,Robots don't show up
Minggu,,,,
1,25,113,55,3
2,2,72,53,0
3,11,68,40,3
4,0,11,9,0
5,0,0,0,0


**Batas QC:** percakapan terakhir 22 October 2026; tidak ada klasifikasi untuk 23-31 Oktober. Jangan atribusikan penurunan di tanggal tersebut pada kategori QC yang belum diperiksa.

**Minggu 4 (22-28 Okt):** CSAT turun 0.32 poin dibanding minggu sebelumnya; 20 catatan QC tersedia. Klasifikasi: Chat Customer No Clear: 11, Information Not Meet Customer Expectations: 9. Ini korelasi deskriptif, bukan penyebab yang terkonfirmasi.

**Tiga tanggal CSAT terendah** (periksa volume dan QC sebelum menyimpulkan):

,Date,Respondents After,CSAT
64,2026-10-03,96.0,3.312
88,2026-10-27,38.0,3.658
74,2026-10-13,27.0,3.667


### Action plan untuk penurunan

1. **Audit minggu 22-28 Oktober dan tanggal rendah.** Cocokkan ID chat bad survey dengan log robot, termasuk 3, 27, dan 28 Oktober. Verifikasi kesalahan jawaban, kendala akses robot, dan topik pelanggan yang tidak terpenuhi. Jangan klaim penyebab pasti tanpa log insiden atau QC percakapan terkait.
2. **Lengkapi QC sampai 31 Oktober.** File QC dapat tertinggal dari ekspor rating; klasifikasi per minggu hanya mewakili percakapan yang sudah diperiksa.
3. **Perbaiki kasus terverifikasi.** Perbarui jawaban/knowledge base dan aturan eskalasi untuk topik berulang, uji ulang dengan percakapan gagal, lalu minta review QC.
4. **Ukur dampak setelah perbaikan.** Pantau CSAT tertimbang mingguan, jumlah responden, dan proporsi bad survey dengan definisi after takeout yang sama. Bandingkan dengan minggu 15-21 Oktober (4,33) dan catat volume sampel.

## Distribusi 10 Sub Category Bad Survey terbanyak, Oktober 2026

Sumber: `dataset_kula/case_handled_oct_2026_source.csv`. Hanya percakapan Oktober bertipe `Bad Survey` dengan `Category` bukan `No Category` **dan** `Sub Category` bukan `Chit-chat`. Persentase tiap `Sub Category` dihitung dari total kasus yang lolos filter, bukan dari seluruh Bad Survey dan bukan tingkat penanganan manusia. Grafik menampilkan 10 subkategori teratas; tabel tetap memuat semua subkategori. Cakupan percakapan dalam sumber hanya sampai 29 Oktober.

In [13]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
import plotly.graph_objects as go

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'dataset_kula' / 'case_handled_oct_2026_source.csv').exists())
case_file = root / 'dataset_kula' / 'case_handled_oct_2026_source.csv'
cases = pd.read_csv(case_file, low_memory=False)
cases['Conversation Start Time'] = pd.to_datetime(
    cases['Conversation Start Time'], errors='coerce')
case_oct = cases[(cases['Conversation Start Time'].dt.year == 2026) &
                 (cases['Conversation Start Time'].dt.month == 10) &
                 (cases['Type'] == 'Bad Survey')].copy()
if case_oct.empty:
    raise ValueError('Tidak ada Bad Survey Oktober 2026 dalam Case Handled')
if case_oct['ID Chat'].isna().any() or case_oct['ID Chat'].duplicated().any():
    raise ValueError('ID Chat kosong/duplikat: periksa data sebelum menghitung persentase')
subcategory_col = 'Sub Category\n(kolom R)'
category = case_oct['Category'].astype('string').str.strip()
subcategory = case_oct[subcategory_col].astype('string').str.strip()
keep = (category.ne('No Category') & subcategory.ne('Chit-chat')).fillna(False)
case_filtered = case_oct.loc[keep].copy()
case_filtered['Sub Category'] = subcategory.loc[keep]
if case_filtered.empty:
    raise ValueError('Tidak ada Bad Survey setelah filter Category dan Sub Category')
case_counts = case_filtered['Sub Category'].value_counts().rename_axis('Sub Category').reset_index(name='kasus')
case_counts['persen'] = case_counts['kasus'].div(len(case_filtered)).mul(100)
assert case_counts['kasus'].sum() == len(case_filtered)
first_day = case_oct['Conversation Start Time'].dt.day.min()
last_day_case = case_oct['Conversation Start Time'].dt.day.max()
display(Markdown(f'**Cakupan percakapan:** {first_day}-{last_day_case} Oktober 2026. '
                 f'{len(case_filtered):,} dari {len(case_oct):,} Bad Survey unik lolos filter '
                 '(Category bukan No Category dan Sub Category bukan Chit-chat). '
                 'Tidak mencakup percakapan 30-31 Oktober.'))
display(case_counts.round({'persen': 2}))

**Cakupan percakapan:** 1-29 Oktober 2026. 267 dari 602 Bad Survey unik lolos filter (Category bukan No Category dan Sub Category bukan Chit-chat). Tidak mencakup percakapan 30-31 Oktober.

,Sub Category,kasus,persen
0,Loan application is rejected,57,21.35
1,Merchant shipping complaints,19,7.12
2,Can't make payments,15,5.62
3,Paylater submission failed,14,5.24
4,Loan bills do not appear,11,4.12
...,...,...,...
83,Cannot buy phone balance with installment,1,0.37
84,Bill notification appears but the bill has bee...,1,0.37
85,Facial verification constraints,1,0.37
86,Can't pay Akulaku PayLater bills via BCA,1,0.37


In [14]:
chart_rows = case_counts.head(10).iloc[::-1]
fig_case = go.Figure()
fig_case.add_trace(go.Bar(y=chart_rows['Sub Category'], x=[100] * len(chart_rows),
    orientation='h', marker_color='#e5eef0', hoverinfo='skip', showlegend=False))
fig_case.add_trace(go.Bar(y=chart_rows['Sub Category'], x=chart_rows['persen'],
    orientation='h', marker_color='#106b62', customdata=chart_rows['kasus'],
    text=chart_rows['persen'].map(lambda value: f'{value:.1f}%'),
    textposition='outside', cliponaxis=False,
    hovertemplate='%{y}: %{customdata:,d} kasus (%{x:.2f}%)<extra></extra>',
    showlegend=False))
fig_case.update_layout(
    title='Top 10 Sub Category Handled by KULA | Oct 2026',
    template='plotly_white', barmode='overlay',
    height=max(470, 31 * len(chart_rows) + 150),
    width=1000, margin=dict(l=370, r=25, t=80, b=80),
    xaxis=dict(title='File source: QC Summary - Bad Survey', range=[0, 111],
               ticksuffix='%', showgrid=True, gridcolor='#edf1f2'),
    yaxis=dict(title='', automargin=True))
fig_case.show(renderer='plotly_mimetype')